# DW Convergence Analysis

Serial `run_markov_circuit` scans for a domain-wall geometry at fixed size and cycle count.
The notebook runs three cases:

1. post-selection
2. perfect correction
3. default adaptive dynamics

All histories are saved under `dw_convergence/cache/`.


## Runtime And CPU Allocation

Set CPU affinity and BLAS thread caps before the simulation imports. The runs are serial over samples (`parallelize_samples=False`), so this block is mainly for thread control and reproducibility.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'dw_convergence' else Path.cwd().resolve()
if not (REPO_ROOT / 'src' / 'fgtn' / 'classA_U1FGTN.py').exists():
    raise RuntimeError(f'Could not locate repo root from cwd={Path.cwd()}')

DW_CONV_DIR = REPO_ROOT / 'dw_convergence'
CACHE_DIR = DW_CONV_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CPU_CAP = 20
AFFINITY_START = 50

# Thread caps for dense linear algebra in serial sample mode.
for env_name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[env_name] = str(CPU_CAP)

if AFFINITY_START is not None:
    try:
        os.sched_setaffinity(0, set(range(AFFINITY_START, AFFINITY_START + CPU_CAP)))
        print(f'CPU affinity set to cores [{AFFINITY_START}, {AFFINITY_START + CPU_CAP - 1}]')
    except Exception as exc:
        print(f'[warn] Could not set CPU affinity: {exc}')
else:
    print('CPU affinity unchanged')

print(f'REPO_ROOT   = {REPO_ROOT}')
print(f'CACHE_DIR   = {CACHE_DIR}')
print(f'CPU_CAP     = {CPU_CAP}')


## Simulation Setup

These are the shared DW and circuit parameters for all three runs.


In [ ]:
import sys
import importlib

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

classA_module = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_module)
classA_U1FGTN = classA_module.classA_U1FGTN

Nx, Ny = 12, 16
nshell = None
cycles = 20
samples = 10
alpha_1, alpha_2 = 30.0, 1.0
sequence = 'dw_symmetric_random'
init_mode = 'default'
save_init = True
save_history_stride = None
progress = True

RUN_CONFIGS = {
    'postselect': {
        'postselect': True,
        'perfect_correction': False,
    },
    'perfect_correction': {
        'postselect': False,
        'perfect_correction': True,
    },
    'default': {
        'postselect': False,
        'perfect_correction': False,
    },
}

print(f'Nx={Nx}, Ny={Ny}, cycles={cycles}, samples={samples}, sequence={sequence}, init_mode={init_mode}')


## Serial DW Runs And History Saving

Each run first computes the expected cache filepath under `dw_convergence/cache/N12x16/`.
If that history file already exists and contains a valid `G_hist`, the notebook skips `run_markov_circuit`, records the cached path, and lets the later analysis cells load from disk as usual.
Otherwise it runs `run_markov_circuit` with `G_history=True` and `parallelize_samples=False` and saves the new history to that expected path.


In [ ]:

import types
import numpy as np


RESULTS = {}
SAVE_PATHS = {}
CACHE_STATUS = {}
SIZE_CACHE_DIR = CACHE_DIR / f'N{Nx}x{Ny}'
SIZE_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _markov_effective_samples(requested_samples, run_kwargs):
    return 1 if bool(run_kwargs.get('postselect', False)) else int(requested_samples)


def _markov_cache_key(*, nx, ny, cycles, samples_count, nshell, dw, init_mode, n_a, seq_mode, exclude, ps, pc):
    nsh = 'None' if nshell is None else str(nshell)
    ex_str = str(exclude) if exclude is not None else 'None'
    key = (
        f'N{int(nx)}x{int(ny)}'
        f'_C{int(cycles)}'
        f'_S{int(samples_count)}'
        f'_nsh{nsh}'
        f'_DW{int(bool(dw))}'
        f'_init-{init_mode}'
        f'_n_a{n_a}'
        f'_seq-{seq_mode}'
        f'_excl{ex_str}'
        f'_ps{int(ps)}'
        f'_pc{int(pc)}'
        '_markov_circuit'
    )
    return key


def _expected_markov_history_path(model, run_kwargs):
    postselect = bool(run_kwargs.get('postselect', False))
    perfect_correction = bool(run_kwargs.get('perfect_correction', False))

    is_dw_active = getattr(model, 'DW', False)
    dw_exclude = None
    dw_exclude_norm = model._normalize_dw_exclude(dw_exclude)
    exclude_arg = dw_exclude_norm if (is_dw_active and dw_exclude_norm is not None) else None

    seq_info = model._sequence_helper(
        sequence,
        Nx=model.Nx,
        Ny=model.Ny,
        rng=np.random.default_rng(0),
        dw_exclude=exclude_arg,
    )
    sequence_mode = seq_info['mode']
    samples_count = _markov_effective_samples(samples, run_kwargs)

    key = _markov_cache_key(
        nx=model.Nx,
        ny=model.Ny,
        cycles=cycles,
        samples_count=samples_count,
        nshell=model.nshell,
        dw=model.DW,
        init_mode=init_mode,
        n_a=0.5,
        seq_mode=sequence_mode,
        exclude=exclude_arg,
        ps=postselect,
        pc=perfect_correction,
    )
    return Path(model._g_history_outdir_rel()) / f'{key}.npz'


def _cached_history_metadata(path):
    if path is None:
        return False, {'reason': 'path is None'}
    path = Path(path)
    if not path.exists():
        return False, {'reason': 'missing file', 'path': str(path)}

    try:
        with np.load(path) as data:
            if 'G_hist' not in data:
                return False, {
                    'reason': 'missing G_hist',
                    'path': str(path),
                    'keys': list(data.keys()),
                }
            shape = tuple(np.asarray(data['G_hist']).shape)
    except Exception as exc:
        return False, {'reason': f'load failed: {exc}', 'path': str(path)}

    if len(shape) != 4:
        return False, {
            'reason': f'unexpected G_hist rank {len(shape)}',
            'path': str(path),
            'shape': shape,
        }

    return True, {
        'path': str(path),
        'shape': shape,
        'samples': int(shape[0]),
        'T': int(shape[1]),
    }


for run_name, run_kwargs in RUN_CONFIGS.items():
    print(f'\n=== Preparing {run_name} ===')
    model = classA_U1FGTN(Nx, Ny, nshell=nshell, DW=True, alpha_1=alpha_1, alpha_2=alpha_2)

    # Redirect built-in cache saving into dw_convergence/cache.
    model._g_history_outdir = types.MethodType(lambda self: self._ensure_outdir(str(SIZE_CACHE_DIR)), model)
    model._g_history_outdir_rel = types.MethodType(lambda self: str(SIZE_CACHE_DIR), model)

    expected_path = _expected_markov_history_path(model, run_kwargs)
    SAVE_PATHS[run_name] = expected_path
    ok, meta = _cached_history_metadata(expected_path)

    if ok:
        CACHE_STATUS[run_name] = 'loaded'
        RESULTS[run_name] = {
            'save_path': str(expected_path),
            'samples': meta['samples'],
            'T': meta['T'],
            'G_hist_shape': meta['shape'],
            'loaded_from_cache': True,
        }
        print(f'{run_name}: using cached history at {expected_path}')
        print(f"{run_name}: G_hist.shape={meta['shape']}")
        continue

    CACHE_STATUS[run_name] = 'computed'
    print(f'{run_name}: cache miss or invalid cache at {expected_path}')
    print(f"{run_name}: reason={meta.get('reason')}")

    res = model.run_markov_circuit(
        G_history=True,
        progress=progress,
        cycles=cycles,
        samples=samples,
        parallelize_samples=False,
        n_jobs=1,
        backend='loky',
        init_mode=init_mode,
        save=True,
        save_init=save_init,
        save_history_stride=save_history_stride,
        sequence=sequence,
        **run_kwargs,
    )

    save_path = Path(res.get('save_path')) if res.get('save_path') is not None else expected_path
    SAVE_PATHS[run_name] = save_path
    RESULTS[run_name] = {
        'save_path': str(save_path),
        'samples': int(res.get('samples', _markov_effective_samples(samples, run_kwargs))),
        'T': int(res.get('T', -1)),
        'loaded_from_cache': False,
    }

    if save_path != expected_path:
        print(f'[warn] {run_name}: expected cache path {expected_path} but run saved to {save_path}')
    else:
        print(f'{run_name}: saved to {save_path}')

print('\nResolved history paths:')
for run_name, path in SAVE_PATHS.items():
    status = CACHE_STATUS.get(run_name, 'unknown')
    print(f'  {run_name}: {path} [{status}]')


## Quick Saved-File Check

This verifies that each saved history exists and reports the stored array shape.


In [ ]:
import numpy as np

for run_name, path in SAVE_PATHS.items():
    if path is None or not path.exists():
        print(f'{run_name}: missing save path')
        continue
    with np.load(path) as data:
        keys = list(data.keys())
        shape = data['G_hist'].shape if 'G_hist' in data else None
    print(f'{run_name}: keys={keys}, G_hist.shape={shape}')


## Convergence Observables From Saved Histories

This block loads the saved histories once and compares three trajectory-averaged observables:

- half-system entropy for the lower half in `y`
- integrated half-system contour summed over both DW boundary columns
- Frobenius distance to the run-specific sample-averaged final state


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _load_histories(save_paths):
    out = {}
    for run_name, path in save_paths.items():
        if path is None or not path.exists():
            raise FileNotFoundError(f'{run_name}: missing history file {path}')
        with np.load(path) as data:
            if 'G_hist' not in data:
                raise KeyError(f"{run_name}: expected 'G_hist' in {path}, found {list(data.keys())}")
            out[run_name] = np.asarray(data['G_hist'], dtype=np.complex128)
    return out


def _bottom_half_indices(nx, ny):
    idx = []
    y_vals = np.arange(0, ny // 2, dtype=int)
    for y in y_vals:
        base = 2 * nx * y
        for x in range(nx):
            idx.append(base + 2 * x)
            idx.append(base + 2 * x + 1)
    return np.asarray(idx, dtype=int), y_vals


def _entanglement_contour_batch(G_batch, nx, ny_sub):
    arr = np.asarray(G_batch, dtype=np.complex128)
    if arr.ndim != 3:
        raise ValueError(f'Expected (B,N,N), got {arr.shape}')
    B, Nlayer, _ = arr.shape
    I = np.eye(Nlayer, dtype=np.complex128)
    G2 = 0.5 * (I + arr)
    evals, vecs = np.linalg.eigh(G2)
    evals = np.clip(np.real_if_close(evals), 1e-12, 1 - 1e-12)
    f = -(evals * np.log(evals) + (1.0 - evals) * np.log(1.0 - evals))
    diagF = np.einsum('bik,bk,bik->bi', vecs, f, vecs.conj(), optimize=True).real
    diagF = diagF.reshape(B, 2, nx, ny_sub, order='F')
    return diagF.sum(axis=1)


def _stderr(a, axis=0):
    n = a.shape[axis]
    if n <= 1:
        return np.zeros(a.shape[1-axis], dtype=float)
    return np.std(a, axis=axis, ddof=1) / np.sqrt(n)


HISTS = _load_histories(SAVE_PATHS)
sub_idx, y_keep = _bottom_half_indices(Nx, Ny)
ny_sub = len(y_keep)

probe_model = classA_U1FGTN(Nx, Ny, nshell=nshell, DW=True, alpha_1=alpha_1, alpha_2=alpha_2)
x0 = int(probe_model.DW_loc[0]) % Nx
x1 = int(probe_model.DW_loc[1]) % Nx
left_pair = np.array([x0 % Nx, (x0 + 1) % Nx], dtype=int)
right_pair = np.array([(x1 - 1) % Nx, x1 % Nx], dtype=int)

OBS = {}
for run_name, G_hist in HISTS.items():
    S, T, N, _ = G_hist.shape
    G_sub_hist = G_hist[:, :, sub_idx][:, :, :, sub_idx]

    half_entropy = np.zeros((S, T), dtype=float)
    boundary_contour = np.zeros((S, T), dtype=float)

    G_final_mean = np.mean(G_hist[:, -1], axis=0)
    fro_dist = np.linalg.norm(G_hist - G_final_mean[None, None, :, :], ord='fro', axis=(2, 3))

    for t in range(T):
        s_map = _entanglement_contour_batch(G_sub_hist[:, t], Nx, ny_sub)
        half_entropy[:, t] = np.sum(s_map, axis=(1, 2))
        boundary_contour[:, t] = (
            np.sum(s_map[:, left_pair, :], axis=(1, 2)) +
            np.sum(s_map[:, right_pair, :], axis=(1, 2))
        )

    OBS[run_name] = {
        't': np.arange(T, dtype=int),
        'half_entropy_mean': np.mean(half_entropy, axis=0),
        'half_entropy_err': _stderr(half_entropy, axis=0),
        'boundary_mean': np.mean(boundary_contour, axis=0),
        'boundary_err': _stderr(boundary_contour, axis=0),
        'fro_mean': np.mean(fro_dist, axis=0),
        'fro_err': _stderr(fro_dist, axis=0),
        'samples': S,
    }

colors = {
    'postselect': '#d62728',
    'perfect_correction': '#1f77b4',
    'default': '#2ca02c',
}
labels = {
    'postselect': 'Post-selection',
    'perfect_correction': 'Perfect correction',
    'default': 'Default adaptive',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

for run_name, obs in OBS.items():
    c = colors.get(run_name, None)
    lbl = f"{labels.get(run_name, run_name)} (S={obs['samples']})"
    t = obs['t']
    axes[0].errorbar(t, obs['half_entropy_mean'], yerr=obs['half_entropy_err'], marker='o', ms=3, lw=1.2, capsize=2, color=c, label=lbl)
    axes[1].errorbar(t, obs['boundary_mean'], yerr=obs['boundary_err'], marker='o', ms=3, lw=1.2, capsize=2, color=c, label=lbl)
    axes[2].errorbar(t, obs['fro_mean'], yerr=obs['fro_err'], marker='o', ms=3, lw=1.2, capsize=2, color=c, label=lbl)

axes[0].set_title('Half-system entropy')
axes[0].set_xlabel('saved time index')
axes[0].set_ylabel('trajectory-avg entropy')
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

axes[1].set_title('DW boundary integrated contour')
axes[1].set_xlabel('saved time index')
axes[1].set_ylabel('left + right boundary sum')
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)

axes[2].set_title('Distance to mean final state')
axes[2].set_xlabel('saved time index')
axes[2].set_ylabel(r'trajectory-avg $\|G_t-\bar G_{\mathrm{final}}\|_F$')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.25, which='both')
axes[2].legend(fontsize=8)

plt.show()

print(f'DW boundary pairs: left={left_pair.tolist()}, right={right_pair.tolist()}')


In [ ]:
def _modular_spectrum_batch(G_batch):
    arr = np.asarray(G_batch, dtype=np.complex128)
    if arr.ndim != 3:
        raise ValueError(f'Expected (B,N,N), got {arr.shape}')
    B, Nlayer, _ = arr.shape
    I = np.eye(Nlayer, dtype=np.complex128)
    C = 0.5 * (I + arr)
    C = 0.5 * (C + np.swapaxes(C.conj(), 1, 2))
    occ = np.linalg.eigvalsh(C)
    occ = np.clip(np.real_if_close(occ), 1e-12, 1.0 - 1e-12)
    eps = np.log((1.0 - occ) / occ)
    return np.sort(np.real_if_close(eps), axis=1)


def _wasserstein_1_equal_weight(spec_batch, ref_batch):
    spec_batch = np.asarray(spec_batch, dtype=float)
    ref_batch = np.asarray(ref_batch, dtype=float)
    if spec_batch.shape != ref_batch.shape:
        raise ValueError(f'Shape mismatch: {spec_batch.shape} vs {ref_batch.shape}')
    return np.mean(np.abs(spec_batch - ref_batch), axis=1)


SPEC_CONV = {}
for run_name, G_hist in HISTS.items():
    S, T, N, _ = G_hist.shape
    G_sub_hist = G_hist[:, :, sub_idx][:, :, :, sub_idx]

    eps_final = _modular_spectrum_batch(G_sub_hist[:, -1])
    w1 = np.zeros((S, T), dtype=float)

    for t in range(T):
        eps_t = _modular_spectrum_batch(G_sub_hist[:, t])
        w1[:, t] = _wasserstein_1_equal_weight(eps_t, eps_final)

    SPEC_CONV[run_name] = {
        't': np.arange(T, dtype=int),
        'w1_mean': np.mean(w1, axis=0),
        'w1_err': _stderr(w1, axis=0),
        'samples': S,
    }

fig, ax = plt.subplots(1, 1, figsize=(8.0, 4.8), constrained_layout=True)

for run_name, obs in SPEC_CONV.items():
    c = colors.get(run_name, None)
    lbl = f"{labels.get(run_name, run_name)} (S={obs['samples']})"
    ax.errorbar(
        obs['t'],
        obs['w1_mean'],
        yerr=obs['w1_err'],
        marker='o',
        ms=3,
        lw=1.2,
        capsize=2,
        color=c,
        label=lbl,
    )

ax.set_title('Half-system entanglement-spectrum Wasserstein convergence')
ax.set_xlabel('saved time index')
ax.set_ylabel(r'trajectory-avg $W_1(\varepsilon_t, \varepsilon_{\mathrm{final}})$')
ax.set_yscale('log')
ax.grid(alpha=0.25, which='both')
ax.legend(fontsize=8)
plt.show()

print(f'Half-system cut uses x=[0,{Nx}) and y=[0,{Ny // 2}) with subsystem dimension {sub_idx.size}.')
print('Wasserstein metric is the equal-weight W1 distance between sorted modular spectra at time t and the final saved time.')


In [ ]:
eq_model = classA_U1FGTN(Nx, Ny, nshell=nshell, DW=True, alpha_1=alpha_1, alpha_2=alpha_2)
G_eq = np.asarray(eq_model.G_CI_domain_wall(periodic=True, triv_region_local_mode=False), dtype=np.complex128)

EQ_DIST = {}
for run_name, G_hist in HISTS.items():
    S, T, N, _ = G_hist.shape
    fro_eq = np.linalg.norm(G_hist - G_eq[None, None, :, :], ord='fro', axis=(2, 3))
    EQ_DIST[run_name] = {
        't': np.arange(T, dtype=int),
        'fro_mean': np.mean(fro_eq, axis=0),
        'fro_err': _stderr(fro_eq, axis=0),
        'samples': S,
    }

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True, sharey=True)
run_order = ['postselect', 'perfect_correction', 'default']

for ax, run_name in zip(axes, run_order):
    obs = EQ_DIST[run_name]
    c = colors.get(run_name, None)
    ax.errorbar(
        obs['t'],
        obs['fro_mean'],
        yerr=obs['fro_err'],
        marker='o',
        ms=3,
        lw=1.2,
        capsize=2,
        color=c,
    )
    ax.set_title(labels.get(run_name, run_name))
    ax.set_xlabel('saved time index')
    ax.grid(alpha=0.25, which='both')
    ax.set_yscale('linear')

axes[0].set_ylabel(r'trajectory-avg $\|G_t - G_{\mathrm{eq}}\|_F$')
plt.show()

print(f'Equilibrium DW covariance shape: {G_eq.shape}')
print(f'Equilibrium DW locations: {getattr(eq_model, "DW_loc", None)}')


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8.0, 4.8), constrained_layout=True)

for run_name, G_hist in HISTS.items():
    c = colors.get(run_name, None)
    samples_count = G_hist.shape[0]
    fro_norm = np.linalg.norm(G_hist, ord='fro', axis=(2, 3))
    fro_mean = np.mean(fro_norm, axis=0)
    fro_err = _stderr(fro_norm, axis=0)
    t = np.arange(G_hist.shape[1], dtype=int)
    lbl = f"{labels.get(run_name, run_name)} (S={samples_count})"
    ax.errorbar(
        t,
        fro_mean,
        yerr=fro_err,
        marker='o',
        ms=3,
        lw=1.2,
        capsize=2,
        color=c,
        label=lbl,
    )

ax.set_title('Trajectory-averaged Frobenius norm of $G_t$')
ax.set_xlabel('saved time index')
ax.set_ylabel(r'trajectory-avg $\|G_t\|_F$')
ax.set_yscale('log')
ax.grid(alpha=0.25, which='both')
ax.legend(fontsize=8)
plt.show()


In [ ]:
x_vals = np.atleast_1d(np.arange(Nx // 2, dtype=int)).astype(int)
run_order = ['postselect', 'perfect_correction', 'default']


def _normalized_wf_site(wf, x, y):
    chi = np.asarray(wf[:, x, y], dtype=np.complex128)
    norm = np.linalg.norm(chi)
    if np.isfinite(norm) and norm > 0:
        chi = chi / norm
    return chi


def _x_resolved_avg_defect_density(G_hist, model, x_vals):
    arr = np.asarray(G_hist, dtype=np.complex128)
    x_vals = np.atleast_1d(np.asarray(x_vals, dtype=int)).reshape(-1)
    if arr.ndim != 4:
        raise ValueError(f'Expected (S,T,N,N), got {arr.shape}')

    S, T, N, N2 = arr.shape
    expected_n = 2 * model.Nx * model.Ny
    if N != N2 or N != expected_n:
        raise ValueError(f'Expected (S,T,{expected_n},{expected_n}), got {arr.shape}')

    have_ow = all(hasattr(model, attr) for attr in ('WF_Ap', 'WF_Bp', 'WF_Am', 'WF_Bm'))
    if not have_ow:
        model.construct_OW_projectors(nshell=model.nshell, DW=model.DW)

    eye = np.eye(N, dtype=np.complex128)
    C_hist = 0.5 * (arr + eye[None, None, :, :])
    D_xt = np.zeros((S, T, len(x_vals)), dtype=float)

    for ix, x in enumerate(x_vals):
        upper_modes = np.column_stack(
            [_normalized_wf_site(model.WF_Ap, int(x), y) for y in range(model.Ny)] +
            [_normalized_wf_site(model.WF_Bp, int(x), y) for y in range(model.Ny)]
        )
        lower_modes = np.column_stack(
            [_normalized_wf_site(model.WF_Am, int(x), y) for y in range(model.Ny)] +
            [_normalized_wf_site(model.WF_Bm, int(x), y) for y in range(model.Ny)]
        )

        occ_upper = np.einsum('um,stuv,vm->stm', upper_modes.conj(), C_hist, upper_modes, optimize=True).real
        occ_lower = np.einsum('um,stuv,vm->stm', lower_modes.conj(), C_hist, lower_modes, optimize=True).real
        D_xt[:, :, ix] = np.mean(occ_upper - occ_lower + 1.0, axis=2)

    return D_xt


probe_model = classA_U1FGTN(Nx, Ny, nshell=nshell, DW=True, alpha_1=alpha_1, alpha_2=alpha_2)
D_eq_x = _x_resolved_avg_defect_density(G_eq[None, None, :, :], probe_model, x_vals)[0, 0]

defect_profiles = {}
for run_name, G_hist in HISTS.items():
    D_xt = np.abs(_x_resolved_avg_defect_density(G_hist, probe_model, x_vals) - D_eq_x[None, None, :])

    mean = np.asarray(np.mean(D_xt, axis=0))
    stderr = np.asarray(_stderr(D_xt, axis=0))

    if mean.ndim == 1:
        mean = mean[:, None]
    if stderr.ndim == 1:
        stderr = stderr[:, None]

    if stderr.shape[1] == 1 and mean.shape[1] > 1:
        stderr = np.repeat(stderr, mean.shape[1], axis=1)

    defect_profiles[run_name] = {
        't': np.arange(D_xt.shape[1], dtype=int),
        'mean': mean,
        'stderr': stderr,
        'samples': D_xt.shape[0],
    }

discrete_colors = list(plt.get_cmap('tab10').colors[:len(x_vals)])
markers = ['o', 's', '^', 'D', 'v', 'P']

fig, axes = plt.subplots(1, 3, figsize=(17.0, 4.8), constrained_layout=True, sharey=True)

for ax, run_name in zip(axes, run_order):
    prof = defect_profiles[run_name]
    for ix, x in enumerate(np.atleast_1d(np.asarray(x_vals, dtype=int)).reshape(-1)):
        ax.errorbar(
            prof['t'],
            prof['mean'][:, ix],
            yerr=prof['stderr'][:, ix],
            color=discrete_colors[ix % len(discrete_colors)],
            marker=markers[ix % len(markers)],
            ms=3,
            lw=1.4,
            label=f'x={x}',
        )
    ax.set_title(f"{labels.get(run_name, run_name)} (S={prof['samples']})")
    ax.set_xlabel('cycle time')
    ax.grid(alpha=0.25)
    ax.set_yscale('log')

axes[0].set_ylabel(r'trajectory-avg $|D(x) - D_{\mathrm{eq}}(x)|$')
axes[-1].legend(title='x', fontsize=8)
plt.show()

print(r'D(x) = mean_{y,nu}[N(x,y,nu,upper) - N(x,y,nu,lower) + 1]')
print(r'Plotted quantity: |D(x) - D_{\mathrm{eq}}(x)|')
print(f'Equilibrium defect profile over x: {D_eq_x.tolist()}')
print(f'Plotted zero-based x values: {x_vals.tolist()}')
print('Cycle time here is the saved Markov history index; these histories were saved every cycle.')
print(prof['mean'])
